# 07 - Experimentos VAE con W&B

Cuaderno de experimentación destinado al entrenamiento de diversas arquitecturas del modelo VAE empleando conjuntos de datos reales en condiciones de operación normal. Cada experimento configurado a través de este entorno queda habilitado para su posterior evaluación mediante el cuaderno `06_model_evaluation_wandb.ipynb`.

Cabe destacar que este documento no reemplaza al cuaderno 05, el cual se preserva como referencia histórica y base metodológica. En el presente entorno se introducen modificaciones en la arquitectura de la red, las dimensiones de la ventana de procesamiento, el número de canales y los hiperparámetros de entrenamiento.


In [ ]:
from pathlib import Path
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

try:
    import wandb
except ImportError:
    wandb = None

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def find_project_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve().parent.parent,
        Path("/workspace/TFM"),
    ]
    for candidate in candidates:
        if (candidate / "datos" / "brutos").exists() and (candidate / "inference_server").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado el directorio raíz del TFM.")

PROJECT_ROOT = find_project_root()
INFERENCE_ROOT = PROJECT_ROOT / "inference_server"
RAW_DIR = PROJECT_ROOT / "datos" / "brutos"
MODEL_DIR = INFERENCE_ROOT / "models"
ANALYSIS_DIR = PROJECT_ROOT / "datos" / "analisis" / "training_runs"
FIGURE_DIR = PROJECT_ROOT / "datos" / "figuras" / "model_training"
WANDB_DIR = PROJECT_ROOT / "datos" / "analisis" / "wandb"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
WANDB_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RAW_DIR={RAW_DIR}")
print(f"MODEL_DIR={MODEL_DIR}")
print(f"WANDB_DIR={WANDB_DIR}")


## Configuración del experimento

Los parámetros de configuración se definen en el archivo `../config/vae_experiments.json`. Para seleccionar una variante distinta, es necesario modificar el valor de `active_experiment` en el archivo JSON o alterar la variable `EXPERIMENT_NAME` en la siguiente celda de código.


In [ ]:
CONFIG_PATH = INFERENCE_ROOT / "config" / "vae_experiments.json"
config_doc = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# Se mantiene en None para emplear config_doc["active_experiment"].
# Se puede especificar un nombre concreto en este punto para realizar pruebas sin modificar el archivo JSON.
EXPERIMENT_NAME = None
EXPERIMENT_NAME = EXPERIMENT_NAME or config_doc["active_experiment"]

if EXPERIMENT_NAME not in config_doc["experiments"]:
    available = ", ".join(sorted(config_doc["experiments"]))
    raise KeyError(f"Experimento no definido: {EXPERIMENT_NAME}. Opciones disponibles: {available}")

CONFIG = dict(config_doc.get("defaults", {}))
CONFIG.update(config_doc["experiments"][EXPERIMENT_NAME])
CONFIG["experiment_name"] = EXPERIMENT_NAME
CONFIG["config_path"] = str(CONFIG_PATH)

wandb_config = config_doc.get("wandb", {})
USE_WANDB = bool(wandb_config.get("enabled", True))
WANDB_PROJECT = wandb_config.get("project", "tfm-railway-anomaly")
WANDB_ENTITY = wandb_config.get("entity")

RUN_NAME = CONFIG["run_name"]
MODEL_FILENAME = CONFIG["model_filename"]
SEED = int(CONFIG["seed"])
SAMPLE_RATE_HZ = int(CONFIG["sample_rate_hz"])
WINDOW_SIZE = int(CONFIG["window_size"])
WINDOW_STEP = int(CONFIG["window_step"])
BATCH_SIZE = int(CONFIG["batch_size"])
EPOCHS = int(CONFIG["epochs"])
LEARNING_RATE = float(CONFIG["learning_rate"])
LATENT_DIM = int(CONFIG["latent_dim"])
BETA = float(CONFIG["beta"])
THRESHOLD_PERCENTILE = float(CONFIG["threshold_percentile"])
MAX_TRAIN_WINDOWS = int(CONFIG["max_train_windows"])
VALIDATION_RATIO = float(CONFIG["validation_ratio"])
MAX_ABS_G = float(CONFIG["max_abs_g"])
FEATURE_COLUMNS = list(CONFIG["feature_columns"])
OPTIMIZER_NAME = str(CONFIG.get("optimizer", "adam")).lower()
WEIGHT_DECAY = float(CONFIG.get("weight_decay", 0.0))
GRAD_CLIP_NORM = float(CONFIG.get("grad_clip_norm", 0.0))
SCHEDULER_NAME = str(CONFIG.get("scheduler", "none")).lower()
BETA_WARMUP_EPOCHS = int(CONFIG.get("beta_warmup_epochs", 0))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CONFIG_PATH={CONFIG_PATH}")
print(f"EXPERIMENT_NAME={EXPERIMENT_NAME}")
print(json.dumps(CONFIG, indent=2))
print(f"DEVICE={DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
REQUIRED_COLUMNS = ["seq", "timestamp_ms", "acc_x_g", "acc_y_g", "acc_z_g"]
BASE_COLUMNS = ["acc_x_g", "acc_y_g", "acc_z_g"]


def read_capture(path):
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name} carece de las siguientes columnas: {missing}")
    return df.drop_duplicates(subset=["seq"], keep="first").sort_values("seq").reset_index(drop=True)


def add_derived_features(df):
    out = df.copy()
    values = out[BASE_COLUMNS].to_numpy(dtype=np.float32)
    out["acc_magnitude_g"] = np.sqrt(np.sum(values**2, axis=1))
    out["delta_x_g"] = out["acc_x_g"].diff().fillna(0.0)
    out["delta_y_g"] = out["acc_y_g"].diff().fillna(0.0)
    out["delta_z_g"] = out["acc_z_g"].diff().fillna(0.0)
    return out


def quality_summary(df):
    seq = df["seq"].to_numpy(np.int64)
    ts = df["timestamp_ms"].to_numpy(np.int64)
    dseq = np.diff(seq)
    duration_s = (ts[-1] - ts[0]) / 1000 if len(ts) > 1 else np.nan
    return {
        "rows": len(df),
        "lost_packets": int(np.maximum(dseq - 1, 0).sum()),
        "duration_s": duration_s,
        "rate_hz": len(df) / duration_s if duration_s and duration_s > 0 else np.nan,
        "rows_abs_over_8g": int((df[BASE_COLUMNS].abs() > MAX_ABS_G).any(axis=1).sum()),
    }

normal_files = sorted(RAW_DIR.glob("real_movement_*.csv"))
if not normal_files:
    raise FileNotFoundError("No existen capturas real_movement_*.csv disponibles para el entrenamiento.")

normal_dfs = []
quality_rows = []
for path in normal_files:
    df = add_derived_features(read_capture(path))
    normal_dfs.append(df)
    quality_rows.append({"file": path.name, **quality_summary(df)})

quality_df = pd.DataFrame(quality_rows)
display(quality_df)


In [ ]:
def make_windows_from_df(df, window_size, step, max_abs_g):
    values = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    base_values = df[BASE_COLUMNS].to_numpy(dtype=np.float32)
    windows = []
    starts_kept = []
    for start in range(0, len(values) - window_size + 1, step):
        window = values[start:start + window_size]
        base_window = base_values[start:start + window_size]
        if np.isfinite(window).all() and np.max(np.abs(base_window)) <= max_abs_g:
            windows.append(window)
            starts_kept.append(start)
    if not windows:
        return np.empty((0, window_size, len(FEATURE_COLUMNS)), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.asarray(windows, dtype=np.float32), np.asarray(starts_kept, dtype=np.int64)

all_windows = []
window_sources = []
for path, df in zip(normal_files, normal_dfs):
    windows, starts = make_windows_from_df(df, WINDOW_SIZE, WINDOW_STEP, MAX_ABS_G)
    total_possible = max(0, (len(df) - WINDOW_SIZE) // WINDOW_STEP + 1)
    print(path.name, windows.shape, f"dropped_windows={total_possible - len(windows)}")
    all_windows.append(windows)
    window_sources.extend([path.name] * len(windows))

windows_raw = np.concatenate(all_windows, axis=0)
window_sources = np.asarray(window_sources)
print("Total de ventanas normales", windows_raw.shape)


In [ ]:
indices = np.random.permutation(len(windows_raw))
val_n = int(len(indices) * VALIDATION_RATIO)
val_idx = indices[:val_n]
train_idx = indices[val_n:]
if len(train_idx) > MAX_TRAIN_WINDOWS:
    train_idx = np.random.choice(train_idx, size=MAX_TRAIN_WINDOWS, replace=False)

train_raw = windows_raw[train_idx]
val_raw = windows_raw[val_idx]
print("train_raw", train_raw.shape)
print("val_raw", val_raw.shape)


In [ ]:
def fit_standardizer(windows):
    flat = windows.reshape(-1, windows.shape[-1])
    mean = flat.mean(axis=0).astype(np.float32)
    std = flat.std(axis=0).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return {"mean": mean, "std": std}


def transform_standardizer(windows, scaler):
    return ((windows - scaler["mean"]) / scaler["std"]).astype(np.float32)


def to_channels_first(windows):
    return np.transpose(windows, (0, 2, 1)).astype(np.float32)

scaler = fit_standardizer(train_raw)
train_x = to_channels_first(transform_standardizer(train_raw, scaler))
val_x = to_channels_first(transform_standardizer(val_raw, scaler))
print("mean", scaler["mean"])
print("std", scaler["std"])
print("train_x", train_x.shape)


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels, window_size, latent_dim):
        super().__init__()
        self.window_size = window_size
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        with torch.no_grad():
            encoded = self.encoder(torch.zeros(1, in_channels, window_size))
        self.encoded_shape = tuple(encoded.shape[1:])
        flat_dim = int(np.prod(self.encoded_shape))
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.ConvTranspose1d(32, in_channels, kernel_size=7, stride=2, padding=3, output_padding=1),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        if not self.training:
            return mu
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z).view(-1, *self.encoded_shape)
        out = self.decoder(h)
        if out.size(-1) != self.window_size:
            out = F.interpolate(out, size=self.window_size, mode="linear", align_corners=False)
        return out

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x, reconstruction, mu, logvar, beta):
    recon = F.mse_loss(reconstruction, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


In [ ]:
def loader(x, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=BATCH_SIZE, shuffle=shuffle)


def init_wandb_run():
    if not USE_WANDB:
        print("W&B desactivado por configuración.")
        return None
    if wandb is None:
        print("La librería wandb no se encuentra instalada; se procede sin registro remoto.")
        return None
    try:
        return wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=RUN_NAME,
            config=CONFIG,
            dir=str(WANDB_DIR),
            reinit="finish_previous",
            settings=wandb.Settings(
                root_dir=str(WANDB_DIR),
                init_timeout=120,
            ),
        )
    except Exception as exc:
        print(f"No ha sido posible inicializar W&B ({type(exc).__name__}: {exc}). Se procede sin registro remoto.")
        return None


def build_optimizer(model):
    if OPTIMIZER_NAME == "adam":
        return torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if OPTIMIZER_NAME == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if OPTIMIZER_NAME == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    raise ValueError(f"Optimizador no soportado: {OPTIMIZER_NAME}")


def build_scheduler(optimizer):
    if SCHEDULER_NAME == "none":
        return None
    if SCHEDULER_NAME == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    if SCHEDULER_NAME == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, EPOCHS // 3), gamma=0.5)
    raise ValueError(f"Planificador de tasa de aprendizaje no soportado: {SCHEDULER_NAME}")


model = ConvVAE(in_channels=len(FEATURE_COLUMNS), window_size=WINDOW_SIZE, latent_dim=LATENT_DIM).to(DEVICE)
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
train_loader = loader(train_x, shuffle=True)
val_loader = loader(val_x, shuffle=False)
print(f"parameters={sum(p.numel() for p in model.parameters()):,}")
print(f"optimizer={OPTIMIZER_NAME} weight_decay={WEIGHT_DECAY} scheduler={SCHEDULER_NAME} grad_clip_norm={GRAD_CLIP_NORM}")

wandb_run = init_wandb_run()


In [ ]:
def beta_for_epoch(epoch):
    if BETA_WARMUP_EPOCHS <= 0:
        return BETA
    return BETA * min(1.0, epoch / BETA_WARMUP_EPOCHS)


def run_epoch(data_loader, train=True, beta_value=BETA):
    model.train(train)
    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0, "n": 0}
    for (batch,) in data_loader:
        batch = batch.to(DEVICE)
        if train:
            optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(batch)
        loss, recon, kl = vae_loss(batch, reconstruction, mu, logvar, beta_value)
        if train:
            loss.backward()
            if GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        bs = batch.size(0)
        totals["loss"] += float(loss.detach().cpu()) * bs
        totals["recon"] += float(recon.detach().cpu()) * bs
        totals["kl"] += float(kl.detach().cpu()) * bs
        totals["n"] += bs
    return {k: totals[k] / totals["n"] for k in ["loss", "recon", "kl"]}

history = []
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    beta_epoch = beta_for_epoch(epoch)
    train_metrics = run_epoch(train_loader, train=True, beta_value=beta_epoch)
    with torch.no_grad():
        val_metrics = run_epoch(val_loader, train=False, beta_value=beta_epoch)
    lr = optimizer.param_groups[0]["lr"]
    row = {
        "epoch": epoch,
        "beta": beta_epoch,
        "learning_rate": lr,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)
    if wandb_run is not None:
        wandb.log(row)
    print(f"epoch={epoch:03d} beta={beta_epoch:.6g} lr={lr:.3g} train_loss={row['train_loss']:.6f} val_loss={row['val_loss']:.6f} val_recon={row['val_recon']:.6f} val_kl={row['val_kl']:.6f}")
    if scheduler is not None:
        scheduler.step()
print(f"training_seconds={time.perf_counter() - start:.2f}")


In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].set_title("Pérdida total")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_recon"], label="reconstruction")
axes[1].plot(history_df["epoch"], history_df["val_kl"], label="KL")
axes[1].set_title("Términos de validación")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
fig_path = FIGURE_DIR / f"{RUN_NAME}_training_loss.png"
plt.savefig(fig_path, dpi=160)
if wandb_run is not None:
    wandb.log({"training_loss_plot": wandb.Image(str(fig_path))})
plt.show()


In [ ]:
@torch.no_grad()
def score_windows(x):
    model.eval()
    rows = []
    for (batch,) in loader(x, shuffle=False):
        batch = batch.to(DEVICE)
        reconstruction, mu, logvar = model(batch)
        recon = torch.mean((batch - reconstruction) ** 2, dim=(1, 2))
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        score = recon + BETA * kl
        rows.append(torch.stack([recon, kl, score], dim=1).cpu().numpy())
    arr = np.concatenate(rows, axis=0)
    return pd.DataFrame(arr, columns=["reconstruction_error", "kl", "vae_score"])

val_scores = score_windows(val_x)
threshold = float(np.percentile(val_scores["vae_score"], THRESHOLD_PERCENTILE))
print(f"threshold p{THRESHOLD_PERCENTILE}={threshold:.6f}")
display(val_scores.describe())
if wandb_run is not None:
    wandb.log({"threshold": threshold, "val_score_mean": float(val_scores['vae_score'].mean()), "val_score_p99": threshold})


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["reconstruction_error", "kl", "vae_score"]):
    ax.hist(val_scores[col], bins=100)
    if col == "vae_score":
        ax.axvline(threshold, color="red", linestyle="--", label="threshold")
        ax.legend()
    ax.set_title(col)
plt.tight_layout()
fig_path = FIGURE_DIR / f"{RUN_NAME}_validation_score_distribution.png"
plt.savefig(fig_path, dpi=160)
if wandb_run is not None:
    wandb.log({"validation_score_distribution": wandb.Image(str(fig_path))})
plt.show()


In [ ]:
SAVE_MODEL = True
artifact_path = MODEL_DIR / MODEL_FILENAME
summary_path = ANALYSIS_DIR / f"{RUN_NAME}_training_summary.json"

if SAVE_MODEL:
    artifact = {
        "model_state_dict": model.state_dict(),
        "config": {
            "model_type": "ConvVAE",
            "window_size": WINDOW_SIZE,
            "window_step": WINDOW_STEP,
            "sample_rate_hz": SAMPLE_RATE_HZ,
            "latent_dim": LATENT_DIM,
            "beta": BETA,
            "feature_columns": FEATURE_COLUMNS,
            "optimizer": OPTIMIZER_NAME,
            "weight_decay": WEIGHT_DECAY,
            "scheduler": SCHEDULER_NAME,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "beta_warmup_epochs": BETA_WARMUP_EPOCHS,
            "training_files": [p.name for p in normal_files],
            "run_name": RUN_NAME,
        },
        "normalization": {
            "mean": scaler["mean"].tolist(),
            "std": scaler["std"].tolist(),
        },
        "threshold": threshold,
        "threshold_percentile": THRESHOLD_PERCENTILE,
    }
    torch.save(artifact, artifact_path)
    summary = {"config": CONFIG, "artifact_path": str(artifact_path), "threshold": threshold, "history_tail": history[-5:]}
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Modelo guardado en: {artifact_path}")
    print(f"Resumen guardado en: {summary_path}")
    if wandb_run is not None:
        wandb.save(str(artifact_path))
        wandb.save(str(summary_path))
        wandb_run.finish()
else:
    print("SAVE_MODEL=False; el artefacto no ha sido guardado.")


## Siguiente fase

Una vez almacenado el modelo, se debe abrir el cuaderno `06_model_evaluation_wandb.ipynb` y actualizar las siguientes variables:

```python
RUN_NAME = "mismo_nombre_del_experimento"
MODEL_PATH = "models/nombre_modelo.pth"
```

A continuación, se requiere ejecutar la totalidad del cuaderno 06 con el propósito de contrastar el desempeño frente a las etiquetas, el valor eficaz (RMS) y la serie de control.
